# ColliderML Release 1: single-event pipeline walker

Go through the conversion steps as exactly the production converter does, by sharing the same machinery. `_event_record_one` owns per-event logic; this notebook calls it directly and inspects intermediates.

In [ ]:
from pathlib import Path
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

IEV = 1

from mlpf.data.colliderml.reader import iter_event_records
from mlpf.data.colliderml.postprocessing import (
    _event_record_one,
    EventData,
    track_features_cml,
    DEFAULT_CALIBRATION,
)
from mlpf.data.colliderml.clustering import cluster_event
from mlpf.data.target_building import assign_genparticles_to_obj_and_merge, assign_to_recoobj

In [ ]:
# slice out exactly the inputs the converter expects
SOURCE = Path('/mnt/ceph/users/ewulff/data/colliderml/CERN__ColliderML-Release-1')
def _src(sub): return SOURCE / f'ttbar_pu0_{sub}' / 'data' / f'ttbar_pu0_{sub}' / 'train-00001-of-01000.parquet'

# walk the shard exactly as the converter does (chunked, event by event) and keep event IEV
ev = None
for i, rec_ in enumerate(iter_event_records(_src('particles'), _src('tracks'), _src('calo_hits'), _src('tracker_hits'))):
    if i == IEV:
        ev = rec_
        break
assert ev is not None

particles_ev = ev['particles']
tracks_ev    = ev['tracks']
calo_ev      = ev['calo_hits']
tracker_ev   = ev['tracker_hits']

## The full pipeline as `_event_record_one`

The debuggability lever is to run the same routine the converter uses, then probe the record it returns and any intermediate state we deliberately keep alive between steps.

In [ ]:
rec = _event_record_one(
    ev['event_id'],
    particles_ev,
    tracks_ev,
    calo_ev,
    tracker_ev,
    algorithm='bfs_merge',
    merge_frac=0.25,
)
for k, v in rec.items():
    try:
        a = np.asarray(v)
        print(f'{k:20s} X shape={a.shape} dtype={a.dtype}')
    except Exception:
        print(f'{k:20s} {v}')

## Step 1 — truth table & attribution

compute_gen_tables(particles_ev, calo_ev, tracks_ev, calibration) → (gen_features, gp_to_hit, gp_to_track, genref_features). Targets = visible leaf primaries only (`keep` logic in truth.py); genref_features = the measurable set (any deposit or track share, pre-visibility) that genmet/genjet are built from.

In [ ]:
from mlpf.data.colliderml.truth import compute_gen_tables, DEFAULT_CALIBRATION as CAL

gen_features, gp_to_hit, gp_to_track, genref_features = compute_gen_tables(particles_ev, calo_ev, tracks_ev, CAL)
n_gp = len(gen_features['PDG'])
print(f'{n_gp} targets kept')
print(f'  genref (measurable, pre-visibility): {len(genref_features["PDG"])} particles, '
      f'E sum {float(np.asarray(genref_features["energy"]).sum()):.1f} GeV vs target E {float(np.asarray(gen_features["energy"]).sum()):.1f} GeV')
print(f'  gp_to_hit edges: {len(gp_to_hit[0])}')
print(f'  gp_to_track edges: {len(gp_to_track[0])}')
pid = np.abs(gen_features['PDG']).astype(int)
print('particle_id counts:', dict(zip(*np.unique(pid, return_counts=True))))

## Step 2 — tracks

track_features_cml(tracks_ev) derives MLPF-ready perigee features from ACTS.

In [ ]:
tfeat = track_features_cml(tracks_ev)
n_track = len(tfeat['type'])
pt = tfeat['pt']
print(f'{n_track} tracks | pt min {pt.min():.2f} med {np.median(pt):.2f} max {pt.max():.2f} GeV')

## Step 3 — calo-hit features + clustering

The clusterer receives calibrated hit energies; the two-region radii table (25 mm ECAL / 90 mm HCAL) is set in clustering.py.

In [ ]:
hit_x = calo_ev['x'].to_numpy()
hit_y = calo_ev['y'].to_numpy()
hit_z = calo_ev['z'].to_numpy()
hit_e_raw = calo_ev['total_energy'].to_numpy()
hit_det = calo_ev['detector'].to_numpy().astype(np.int64)

calib = np.array([CAL[int(d)] for d in hit_det])
hit_e_cal = hit_e_raw * calib
print(f'{len(hit_x)} hits — calibrated E sum {hit_e_cal.sum():.2f} GeV')

cluster_of, cl_feats, hit_to_cluster, cluster_region = cluster_event(
    hit_x, hit_y, hit_z, hit_e_cal, hit_det, algorithm='bfs_merge', merge_frac=0.25
)
n_cluster = int(cluster_of.max()) + 1
print(f'{n_cluster} clusters (detector regions: {np.unique(cluster_region).tolist()})')

fig, ax = plt.subplots(figsize=(6,4))
ax.hist(cl_feats[:,5], bins=60, color='#4c72b0')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('cluster E [GeV]'); ax.set_title('cluster E spectrum')
plt.show()

## Step 4 — allocator

assign_genparticles_to_obj_and_merge takes the attribution maps (gp_to_hit/gp_to_track) and synthesizes the exclusive representatives: `gp_to_obj`.

In [ ]:
gpdata = EventData(
    gen_features,
    {'type': np.zeros(len(hit_x), dtype=np.float32)},
    {'type': np.zeros(n_cluster, dtype=np.float32)},
    {'type': tfeat['type']},
    gp_to_hit,
    gp_to_track,
    hit_to_cluster,
    (np.array([]), np.array([])),
)

gpdata_cleaned, gp_to_obj, gp_to_hit_idx, trk_incl, cls_incl, hit_incl = assign_genparticles_to_obj_and_merge(gpdata)
n_gp_clean = len(gpdata_cleaned.gen_features['PDG'])
print(f'allocator kept {n_gp_clean} target rows (of {n_gp})')
print(f'  exclusive track-owned: {(gp_to_obj[:,0] != -1).sum()}')
print(f'  exclusive cluster-owned: {(gp_to_obj[:,1] != -1).sum()}')
print(f'  both-exclusive-(-1) rows: {((gp_to_obj[:,0] == -1) & (gp_to_obj[:,1] == -1)).sum()}' if False else f'  both-exclusive-(-1) rows: {((gp_to_obj[:,0] == -1) & (gp_to_obj[:,1] == -1)).sum()}')

## Step 5 — exclusive deposit

R2 splits each truth row's calo energy into the clusters it owns exclusively. Same arithmetic as the converter

In [ ]:
# invert exclusive owner map: cluster -> clean gp
owner_of_cluster = np.full(n_cluster, -1, dtype=np.int64)
for igp in range(n_gp_clean):
    cl_id = int(gp_to_obj[igp, 1])
    if cl_id != -1:
        owner_of_cluster[cl_id] = igp

gp_to_cluster_excl = np.zeros(n_gp_clean, dtype=np.float64)
gpth = gpdata_cleaned.genparticle_to_hit
if len(gpth[0]):
    hit_to_cluster_idx = np.asarray(hit_to_cluster[1], dtype=np.int64)
    hits_attr = np.asarray(gpth[1], dtype=np.int64)
    gps_attr = np.asarray(gpth[0], dtype=np.int64)
    w_attr = np.asarray(gpth[2], dtype=np.float64)
    owner_of_hit = owner_of_cluster[hit_to_cluster_idx[hits_attr]]
    m = (owner_of_hit >= 0) & (owner_of_hit == gps_attr)
    np.add.at(gp_to_cluster_excl, gps_attr[m], w_attr[m])

E = np.asarray(gpdata_cleaned.gen_features['energy'], dtype=np.float64)
ratio = np.divide(gp_to_cluster_excl, E, out=np.zeros_like(E), where=E > 0)
nz = ratio[ratio > 0]
print(f'median exclusive capture fraction: {np.median(nz):.3f}')
print(f'min/max nonzero: {nz.min():.3f}─{nz.max():.3f}')

## Step 6 — 3D display from `rec`

In [ ]:
X_track = np.asarray(rec['X_track'])
X_cluster = np.asarray(rec['X_cluster'])
X_hit_calo = np.asarray(rec['X_hit_calo'])
X_hit_tracker = np.asarray(rec['X_hit_tracker'])
y_track = np.asarray(rec['ytarget_track'])
y_cluster = np.asarray(rec['ytarget_cluster'])
h2c = np.asarray(rec['hit_to_cluster'])

import plotly.graph_objects as go

def cluster_color(e_ecal, e_hcal):
    tot = max(e_ecal + e_hcal, 1e-12)
    if e_ecal / tot > 0.75: return 'rgba(31,119,180,0.7)'
    if e_ecal / tot < 0.25: return 'rgba(214,39,40,0.7)'
    return 'rgba(127,127,127,0.7)'

cl_xyz = X_cluster[:, 6:9]
cl_E = X_cluster[:, 5]
cl_sig = X_cluster[:, 14:17].mean(axis=1)
cl_cols = [cluster_color(x, y) for x, y in zip(X_cluster[:, 10], X_cluster[:, 11])]
sizes = np.clip(2.0 * np.sqrt(np.maximum(cl_E, 0.0)), 2.0, 40.0)

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=X_hit_calo[:,6], y=X_hit_calo[:,7], z=X_hit_calo[:,8],
    mode='markers', name='calo hits',
    marker=dict(size=1.5, color=np.log10(np.maximum(X_hit_calo[:,5], 1e-9)),
                colorscale='Viridis', colorbar=dict(title='log10 E [GeV]'), opacity=0.5)))
if len(X_hit_tracker):
    fig.add_trace(go.Scatter3d(x=X_hit_tracker[:,6], y=X_hit_tracker[:,7], z=X_hit_tracker[:,8],
        mode='markers', name='tracker hits', marker=dict(size=1.0, color='rgba(50,50,200,0.4)')))
fig.add_trace(go.Scatter3d(x=cl_xyz[:,0], y=cl_xyz[:,1], z=cl_xyz[:,2],
    mode='markers', name='clusters', marker=dict(size=sizes, color=cl_cols, opacity=0.6),
    text=[f'E={e:.2f} GeV, σ={s:.0f} mm' for e, s in zip(cl_E, cl_sig)],
    hovertemplate='%{text}<extra>cluster</extra>'))

trk_xs, trk_ys, trk_zs = [], [], []
for i in range(int(np.count_nonzero(X_track[:,0]))):
    phi_i = np.arctan2(X_track[i,3], X_track[i,4])
    x0 = -X_track[i,6] * np.sin(phi_i)
    y0 = X_track[i,6] * np.cos(phi_i)
    z0 = X_track[i,7]
    dx, dy, dz = np.cos(phi_i), np.sin(phi_i), np.sinh(X_track[i,2])
    n = np.sqrt(dx*dx + dy*dy + dz*dz)
    dx, dy, dz = dx/n, dy/n, dz/n
    for d in np.linspace(0, 1250, 2):
        trk_xs.append(x0 + dx * d); trk_ys.append(y0 + dy * d); trk_zs.append(z0 + dz * d)
    trk_xs.append(None); trk_ys.append(None); trk_zs.append(None)
fig.add_trace(go.Scatter3d(x=trk_xs, y=trk_ys, z=trk_zs, mode='lines', name='tracks',
    line=dict(color='rgba(50,50,50,0.8)', width=2)))

# pin the 3D axes to the full-data bounding box (all layers incl. track endpoints): plotly's
# default autorange recomputes over *visible* traces, so legend toggles would re-zoom the camera
_layer_xyz = [X_hit_calo[:, 6:9], cl_xyz]
if len(X_hit_tracker):
    _layer_xyz.append(X_hit_tracker[:, 6:9])
if len(X_track):
    _phi = np.arctan2(X_track[:, 3], X_track[:, 4])
    _ts = np.stack([-X_track[:, 6] * np.sin(_phi), X_track[:, 6] * np.cos(_phi), X_track[:, 7]], axis=1)
    _td = np.stack([np.cos(_phi), np.sin(_phi), np.sinh(X_track[:, 2])], axis=1)
    _td /= np.linalg.norm(_td, axis=1, keepdims=True)
    _layer_xyz += [_ts, _ts + _td * 1250.0]
_pts = np.concatenate([a for a in _layer_xyz if len(a)], axis=0)
_lo, _hi = _pts.min(axis=0), _pts.max(axis=0)
_pad = np.maximum((_hi - _lo) * 0.02, 1.0)
_rng = {a: [float(l), float(h)] for a, l, h in zip(('x', 'y', 'z'), _lo - _pad, _hi + _pad)}

fig.update_layout(
    scene=dict(xaxis=dict(title='x [mm]', range=_rng['x'], autorange=False),
               yaxis=dict(title='y [mm]', range=_rng['y'], autorange=False),
               zaxis=dict(title='z [mm]', range=_rng['z'], autorange=False)),
    scene_camera=dict(eye=dict(x=2.0, y=0.6, z=0.0), up=dict(x=0, y=1, z=0)),
    height=650,
    title=f'ColliderML event {IEV} — hits + tracks + clusters',
    legend=dict(x=0.02, y=0.98, bordercolor='rgba(50,50,50,0.2)', bgcolor='rgba(255,255,255,0.85)'))
fig.show()

## Validator cross-check

Run `tests/validate_parquet.py` against a chosen input — either a single shard or a directory
of them — and print the gate table. The cell fails visibly only if the validator crashes.
`SHOW_PLOTS` renders each gate plot inline below the cell; set `SAVE_PLOTS` to also write the
PNGs + report JSON to `PLOTS_DIR` instead.

In [ ]:
"set INPUT to any parquet file (one event's shard) or a directory of them. notebooks/ is sys.path[0], so add the repo root before importing tests."
import os
import sys
from pathlib import Path

_repo = Path.cwd().resolve()
if not (_repo / "tests" / "validate_parquet.py").exists():
    _repo = _repo.parent
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))

from tests.validate_parquet import ParquetValidator


INPUT = Path(os.environ.get("COLLIDERML_PARQUET", "/mnt/ceph/users/ewulff/data/colliderml/mlpf_parquet/clustered/ttbar_pu0/train-00000-of-01000.parquet"))
MAX_EVENTS = 100
PLOTS_DIR = Path("validation_plots_dodge")  # only used when SAVE_PLOTS is True
SHOW_PLOTS = True   # render the gate plots inline below the cell
SAVE_PLOTS = False  # write per-gate PNGs + validation_report.json into PLOTS_DIR

files = sorted(INPUT.glob("*.parquet")) if INPUT.is_dir() else [INPUT]
print(f"validating {len(files)} shard(s) — up to {MAX_EVENTS} events each")
for f in files:
    val = ParquetValidator(f, "colliderml", MAX_EVENTS, PLOTS_DIR / (f.stem if len(files) > 1 else ""), save_plots=SAVE_PLOTS, show_plots=SHOW_PLOTS)
    val.run()
    n_pass = sum(1 for g in val.gates if g.status == "PASS")
    print(f"\n{f.name}: PASS={n_pass} FAIL={sum(1 for g in val.gates if g.status == 'FAIL')} "
          f"WARN={sum(1 for g in val.gates if g.status == 'WARN')} SKIP={sum(1 for g in val.gates if g.status == 'SKIP')}")
    for g in val.gates:
        symbol = {"PASS": "ok", "FAIL": "FAIL", "WARN": "WARN", "SKIP": "-"}[g.status]
        print(f"  {symbol} {g.gate_id}  {g.title}: {g.observed}")


That runs the full validator gate set on the chosen shard, rendering the per-gate plots
inline (`SHOW_PLOTS=True` above). Flip `SAVE_PLOTS` if you also want the PNGs + report
JSON in `validation_plots_dodge/`. To validate the whole converted corpus, point `INPUT` at
`/mnt/ceph/users/ewulff/data/colliderml/mlpf_parquet/clustered/` and drop the events cap. The
current defaults (`MAX_EVENTS=50`) keep this under a minute per shard; the gate observations
are printed in the stdio above (and stay on `val.gates` in memory).